In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scipy as sp
from scipy.spatial.distance import pdist, squareform
import scipy.stats as stats

# Zadanie 1. 
Wczytaj macierz BIOM z zestawu environmental. Wykorzystując jawny broadcasting, oblicz odległość Euklidesa i Jaccarda między wszystkimi parami próbek w tym zestawie danych. Narysuj klastrowaną mapę ciepła i opisz swoje wnioski. Czy obie odległości wskazują na te same wnioski?

In [ ]:
biom = pd.read_csv("data/environmental/biom.csv")

In [ ]:
biom.index = biom.iloc[:, 0]
biom = biom.drop(columns=biom.columns[0])
biom.index.name = "sample_id"
biom = biom.map(lambda x: 0 if not isinstance(x, int) else x)

In [ ]:
biom_numpy = biom.to_numpy()
n = biom_numpy.shape[0]

# macierze
eu_dist = np.zeros((n, n))
jac_dist = np.zeros((n, n))

for i in range(n):
    exp = biom_numpy[i, np.newaxis, :] 
    
    diff = exp - biom_numpy
    eu_dist[i, :] = np.sqrt(np.sum(diff**2, axis=1))
    
    inter = np.sum(np.minimum(exp, biom_numpy), axis=1)
    togh = np.sum(np.maximum(exp, biom_numpy), axis=1)
    
    with np.errstate(divide="ignore", invalid="ignore"):
        jacc = 1.0 - (inter / togh)
        jacc[np.isnan(jacc)] = 0.0 
        
    jac_dist[i, :] = jacc

eu_dist = (eu_dist + eu_dist.T) / 2.0
jac_dist = (jac_dist + jac_dist.T) / 2.0
np.fill_diagonal(eu_dist, 0.0)
np.fill_diagonal(jac_dist, 0.0)

lim = 150
eu_set = eu_dist[:lim, :lim]
jac_set = jac_dist[:lim, :lim]

# eu
plt.figure(figsize=(10, 10))
sns.clustermap(eu_set, cmap="RdBu", figsize=(8, 8), 
               robust=True, xticklabels=False, yticklabels=False
               )

# jacc
plt.figure(figsize=(10, 10))
sns.clustermap(jac_set, cmap="plasma", figsize=(8, 8), 
               robust=True, xticklabels=False, yticklabels=False
               )

plt.show()

# Zadanie 2.
W zestawie danych environmental każde stanowisko ma 90 próbek. Podziel macierz BIOM na stanowiska wykorzystując wbudowane funkcje NumPy.

In [ ]:
smpls = 90
sites_n = biom_numpy.shape[0] // smpls

In [ ]:
sites_list = np.vsplit(biom_numpy, sites_n)

In [ ]:
print(f"typ wyniku: {type(sites_list)}")
print(f"liczba stanowisk: {len(sites_list)}")
print(f"kształt jednego stanowiska: {sites_list[0].shape}")

In [ ]:
sites_tensor = biom_numpy.reshape(sites_list, smpls, biom_numpy.shape[1])

In [ ]:
print(f"typ wyniku: {type(sites_tensor)}")
print(f"kształt tensora ze stanowiskami: {sites_tensor.shape}")

# Zadanie 3.

Wykorzystując *stacking*, złącz w jedną macierz macierze BIOM z próbek o numerach parzystych. Sprawdź czy średnia liczba wystąpień ASV jest statystycznie istotnie różna od liczby wystąpień powstałej ze złączenia macierzy próbek o numerach nieparzystych.

In [ ]:
parzyste_list = [biom_numpy[i, :] for i in range(0, biom_numpy.shape[0], 2)]
nieparzyste_list = [biom_numpy[i, :] for i in range(1, biom_numpy.shape[0], 2)]

In [ ]:
biom_parzyste = np.vstack(parzyste_list)
biom_nieparzyste = np.vstack(nieparzyste_list)

In [ ]:
mean_parzyste = np.mean(biom_parzyste, axis=0)
mean_nieparzyste = np.mean(biom_nieparzyste, axis=0)

In [ ]:
stat, p_value = stats.wilcoxon(mean_parzyste, mean_nieparzyste)

In [ ]:
print(f"statystyka Wilcoxona: {stat:.2f}")
print(f"p: {p_value:.5e}\n")

In [ ]:
alpha = 0.05
if p_value < alpha:
    print("wniosek: odrzucam hipotezę zerową, jest statystycznie istotna różnica w średniej liczbie wystąpień ASV między parzystymi i nieparzystymi próbkami.")
else:
    print("wniosek: nie ma podstaw do odrzucenia hipotezy zerowej, różnice między parzystymi i nieparzystymi próbkami nie SĄ statystycznie istotne i mogą wynikać z przypadku.")

# Zadanie 4.
Każda próbka w zestawie environmental była wykonywana w pięciu powtórzeniach. Dokonaj transformacji tabeli BIOM w taki sposób aby kolejne powtórzenia były reprezentowane jako kolejne macierze w trzecim wymiarze tensora BIOM. Sprawdź czy istnieją statystycznie istotne różnice w liczbie wystąpień między kolejnymi powtórzeniami.

In [ ]:
n_rows, n_features = biom_numpy.shape
n_rep = 5
n_samples = n_rows // n_rep

In [ ]:
biom_reshaped = biom_numpy.reshape(n_samples, n_rep, n_features)
tensor_biom = np.moveaxis(biom_reshaped, 1, 2)
print(f"kształt tensora BIOM: {tensor_biom.shape}")

In [ ]:
reads_per_rep = np.sum(tensor_biom, axis=1)
rep1, rep2, rep3, rep4, rep5 = reads_per_rep.T

In [ ]:
stat, p_value = stats.friedmanchisquare(rep1, rep2, rep3, rep4, rep5)

In [ ]:
print(f"statystyka friedmana: {stat:.2f}")
print(f"p: {p_value:.5e}")

In [ ]:
alpha = 0.05
if p_value < alpha:
    print("wniosek: odrzucam hipotezę zerową, jest statystycznie istotna różnica w średniej liczbie wystąpień ASV między parzystymi i nieparzystymi próbkami.")
else:
    print("wniosek: nie ma podstaw do odrzucenia hipotezy zerowej, różnice między parzystymi i nieparzystymi próbkami nie SĄ statystycznie istotne i mogą wynikać z przypadku.")

# Zadanie 5.
Używając wyłącznie operacji NumPy oblicz entropię $H_{Shannon}$ z macierzy BIOM.

In [ ]:
row_sums = np.sum(biom_numpy, axis=1, keepdims=True)

with np.errstate(divide="ignore", invalid="ignore"):
    p = biom_numpy / row_sums

log_p = np.log(p, out=np.zeros_like(p), where=(p > 0))

shannon = -np.nansum(p * log_p, axis=1)

print(shannon[:5])